In [ ]:
# TechCorp Industries — Fine-tuning LoRA du modele medical experimental

**Mission IA (experimentale, pas pour production)** : fine-tuner un modele Phi-3.5 sur le dataset medical `ruslanmv/ai-medical-chatbot`, deja telecharge et nettoye par l'equipe DATA (`rendu/data/medical_dataset_train.json` / `medical_dataset_val.json`, 245 941 conversations nettoyees).

**A faire avant de lancer** :
1. Menu Colab -> Modifier -> Parametres du notebook -> GPU (T4 minimum, A100 si Colab Pro).
2. Uploader `medical_dataset_train.json` et `medical_dataset_val.json` sur votre Google Drive (dans un dossier `techcorp/`), puis monter le Drive ci-dessous. Les fichiers font ~230 Mo / ~25 Mo, trop gros pour un upload direct fiable dans Colab.
3. Ce notebook sous-echantillonne le dataset (voir `SAMPLE_TRAIN_SIZE`) pour rester dans un temps d'entrainement raisonnable pendant le hackathon — 245k exemples prendraient des heures meme sur A100. Augmentez si vous avez le temps/GPU.

**Rappel de securite (cf rapport DATA)** : le dataset financier heritee contenait un backdoor de data poisoning. Le dataset medical a ete nettoye separement et n'est pas concerne, mais ce modele reste **experimental** — pas de deploiement en production, et une relecture humaine (medecin) serait necessaire avant tout usage reel (cf `medical_project/Readme.md`).

In [ ]:
!pip install -q transformers==4.45.2 peft==0.13.2 accelerate==0.34.2 bitsandbytes==0.44.1 datasets==2.20.0 matplotlib

In [ ]:
import torch
print('CUDA disponible :', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
!nvidia-smi

## 1. Monter Google Drive et charger le dataset nettoyé

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Adaptez ce chemin a l'emplacement ou vous avez uploade les fichiers DATA
DATA_DIR = '/content/drive/MyDrive/techcorp'
TRAIN_PATH = f'{DATA_DIR}/medical_dataset_train.json'
VAL_PATH = f'{DATA_DIR}/medical_dataset_val.json'

In [ ]:
import json
import random

random.seed(42)

# Sous-echantillonnage : 245k exemples est bien trop pour un hackathon de 7h.
# On prend un echantillon representatif, comme cote financier (~2500 exemples
# avaient suffi a entrainer le modele finance en ~1h45 sur GPU selon
# logs/training.log). Augmentez si le temps/GPU le permettent.
SAMPLE_TRAIN_SIZE = 3000
SAMPLE_VAL_SIZE = 300

with open(TRAIN_PATH, encoding='utf-8') as f:
    train_data = json.load(f)
with open(VAL_PATH, encoding='utf-8') as f:
    val_data = json.load(f)

print('Train disponible :', len(train_data), '| Val disponible :', len(val_data))

random.shuffle(train_data)
random.shuffle(val_data)
train_data = train_data[:SAMPLE_TRAIN_SIZE]
val_data = val_data[:SAMPLE_VAL_SIZE]

print('Train utilise :', len(train_data), '| Val utilise :', len(val_data))
print('\nExemple :', train_data[0])

## 2. Formatage — IMPORTANT : fix du bug present dans `scripts/train_finance_model.py`

Le script d'entrainement herite de l'equipe precedente utilise `item['input']`
comme message utilisateur pour le format `instruction/input/output`, alors
que la vraie question est dans `item['instruction']` (`input` est presque
toujours vide dans nos datasets). Resultat : le modele etait entraine sur des
tours utilisateur vides -> reponse, ce qui degrade fortement l'apprentissage.
On corrige ca ici en utilisant `instruction` en priorite.

In [ ]:
def to_chat_text(item):
    user_msg = (item.get('instruction') or item.get('input') or item.get('question') or '').strip()
    assistant_msg = (item.get('output') or item.get('answer') or '').strip()
    return f"<|user|>\n{user_msg}<|end|>\n<|assistant|>\n{assistant_msg}<|end|>"

train_texts = [{'text': to_chat_text(x)} for x in train_data if x.get('instruction') and x.get('output')]
val_texts = [{'text': to_chat_text(x)} for x in val_data if x.get('instruction') and x.get('output')]

print('Train pret :', len(train_texts), '| Val pret :', len(val_texts))
print('\nExemple formate :\n', train_texts[0]['text'][:400])

## 3. Chargement du modele de base + configuration LoRA

Base recommandee par `medical_project/Readme.md` : **microsoft/Phi-3.5-mini-instruct**
(meme famille Phi-3 que le modele financier -> meme format de chat `<|user|>` / `<|assistant|>`).

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

BASE_MODEL = "microsoft/Phi-3.5-mini-instruct"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["qkv_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 4. Tokenisation

In [ ]:
from datasets import Dataset

def tokenize_fn(examples):
    tokenized = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=512,
        return_tensors="pt",
    )
    tokenized["labels"] = tokenized["input_ids"].clone()
    return tokenized

train_ds = Dataset.from_list(train_texts).map(tokenize_fn, batched=True, remove_columns=["text"])
val_ds = Dataset.from_list(val_texts).map(tokenize_fn, batched=True, remove_columns=["text"])
print(train_ds, val_ds)

## 5. Entrainement (avec evaluation, pour le suivi loss train/val demande dans le brief)

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

OUTPUT_DIR = "/content/drive/MyDrive/techcorp/phi35_medical_lora"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=50,
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=100,
    save_steps=200,
    save_total_limit=2,
    remove_unused_columns=False,
    dataloader_drop_last=True,
    fp16=True,
    report_to="none",
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=data_collator,
)

train_result = trainer.train()
trainer.save_model()
print(train_result)

## 6. Courbe de loss (métriques d'entraînement à partager avec l'équipe)

In [ ]:
import matplotlib.pyplot as plt

history = trainer.state.log_history
train_loss = [(h["step"], h["loss"]) for h in history if "loss" in h]
eval_loss = [(h["step"], h["eval_loss"]) for h in history if "eval_loss" in h]

plt.figure(figsize=(8, 5))
if train_loss:
    steps, losses = zip(*train_loss)
    plt.plot(steps, losses, label="train loss")
if eval_loss:
    steps, losses = zip(*eval_loss)
    plt.plot(steps, losses, label="eval loss", marker="o")
plt.xlabel("step")
plt.ylabel("loss")
plt.title("Fine-tuning LoRA - modele medical experimental")
plt.legend()
plt.savefig(f"{OUTPUT_DIR}/loss_curve.png")
plt.show()

print("epochs:", training_args.num_train_epochs)
print("final train loss:", train_loss[-1][1] if train_loss else None)
print("final eval loss:", eval_loss[-1][1] if eval_loss else None)

## 7. Test rapide du modele fine-tune (tests de performance demandes dans le brief)

In [ ]:
TEST_MEDICAL_PROMPTS = [
    "I have had a persistent headache and mild fever for two days, what could this be?",
    "What are the common side effects of ibuprofen?",
    "Can you explain what high blood pressure means for my health?",
    "I feel very anxious lately and have trouble sleeping, what should I do?",
    "What is the difference between a virus and a bacteria in simple terms?",
]

model.eval()
for prompt in TEST_MEDICAL_PROMPTS:
    inputs = tokenizer(f"<|user|>\n{prompt}<|end|>\n<|assistant|>\n", return_tensors="pt", truncation=True, max_length=512)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.4,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.15,
            pad_token_id=tokenizer.eos_token_id,
        )
    response = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"\n👤 {prompt}\n🤖 {response}\n{'-'*60}")

## 8. A faire ensuite

1. Noter les métriques finales (loss train/val, nombre d'epochs, taille de
   l'échantillon utilisé) dans `rendu/ia/RAPPORT_IA.md`.
2. Partager le lien de ce Colab avec l'équipe (bouton *Share* en haut à
   droite, accès "Anyone with the link - Viewer" suffit).
3. **Rappel** : modèle expérimental, non validé médicalement — ne pas le
   présenter comme utilisable en production, uniquement comme preuve de
   concept R&D pour la présentation orale.